# 04 - Model Comparison: SMS Spam Detection

## Overview
This notebook provides a comprehensive comparative analysis of all implemented models.

### Analysis Includes:
1. Performance metrics comparison
2. ROC curves comparison
3. Confusion matrices summary
4. Training time analysis
5. Trade-offs and recommendations

---

## 1. Import Libraries and Load Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("Libraries imported successfully!")

In [ ]:
# Load results from previous notebook
with open('../results/metrics/all_results.json', 'r') as f:
    results = json.load(f)

# Load comparison table
comparison_df = pd.read_csv('../results/metrics/model_comparison.csv')

print("Results loaded successfully!")
print(f"\nModels evaluated: {list(results.keys())}")

## 2. Overall Performance Comparison

In [ ]:
# Display comparison table
print("\n" + "=" * 90)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 90)
print(comparison_df.to_string(index=False))
print("=" * 90)

In [ ]:
# Prepare data for visualization
models = list(results.keys())
metrics_names = ['accuracy', 'precision', 'recall', 'f1_score']

# Create DataFrame for plotting
plot_data = pd.DataFrame(results).T[metrics_names]
plot_data.index.name = 'Model'
plot_data = plot_data.reset_index()

print("\nData prepared for visualization:")
plot_data

## 3. Visual Comparison: All Metrics

In [ ]:
# Create grouped bar chart for all metrics
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(models))
width = 0.2
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for i, metric in enumerate(metrics_names):
    values = [results[m][metric] for m in models]
    bars = ax.bar(x + i * width, values, width, label=metric.replace('_', ' ').title(), color=colors[i])
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
               f'{value:.3f}', ha='center', va='bottom', fontsize=8, rotation=0)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison: All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/all_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Individual Metric Comparison

In [ ]:
# Create individual metric comparison charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_names):
    ax = axes[idx]
    values = [results[m][metric] for m in models]
    
    # Sort by value for better visualization
    sorted_pairs = sorted(zip(models, values), key=lambda x: x[1], reverse=True)
    sorted_models, sorted_values = zip(*sorted_pairs)
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(models)))
    bars = ax.barh(sorted_models, sorted_values, color=colors)
    
    # Add value labels
    for bar, value in zip(bars, sorted_values):
        ax.text(value + 0.005, bar.get_y() + bar.get_height()/2,
               f'{value:.4f}', va='center', fontsize=10)
    
    ax.set_xlabel('Score', fontsize=11)
    ax.set_title(f'{metric.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    ax.set_xlim(0, max(sorted_values) * 1.1)

plt.suptitle('Individual Metric Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/plots/individual_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. F1-Score Comparison (Primary Metric)

In [ ]:
# F1-Score is the primary metric for imbalanced classification
fig, ax = plt.subplots(figsize=(12, 6))

f1_scores = [results[m]['f1_score'] for m in models]
sorted_pairs = sorted(zip(models, f1_scores), key=lambda x: x[1], reverse=True)
sorted_models, sorted_f1 = zip(*sorted_pairs)

colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(models))]
bars = ax.bar(sorted_models, sorted_f1, color=colors, edgecolor='black', linewidth=1)

# Highlight best model
bars[0].set_color('#27ae60')
bars[0].set_edgecolor('#1e8449')
bars[0].set_linewidth(2)

# Add value labels
for bar, value in zip(bars, sorted_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
           f'{value:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('F1-Score Comparison (Primary Metric)\nBest Model Highlighted in Green', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0, max(sorted_f1) * 1.1)
ax.axhline(y=np.mean(sorted_f1), color='red', linestyle='--', label=f'Average: {np.mean(sorted_f1):.4f}')
ax.legend()

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../results/plots/f1_score_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBest model by F1-Score: {sorted_models[0]} ({sorted_f1[0]:.4f})")

## 6. Training Time Analysis

In [ ]:
# Training time comparison
fig, ax = plt.subplots(figsize=(12, 6))

train_times = [results[m]['training_time'] for m in models]
sorted_pairs = sorted(zip(models, train_times), key=lambda x: x[1])
sorted_models, sorted_times = zip(*sorted_pairs)

colors = plt.cm.coolwarm(np.linspace(0.2, 0.8, len(models)))
bars = ax.barh(sorted_models, sorted_times, color=colors)

# Add value labels
for bar, value in zip(bars, sorted_times):
    ax.text(value + max(sorted_times)*0.02, bar.get_y() + bar.get_height()/2,
           f'{value:.2f}s', va='center', fontsize=10)

ax.set_xlabel('Training Time (seconds)', fontsize=12)
ax.set_ylabel('Model', fontsize=12)
ax.set_title('Training Time Comparison', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/plots/training_time_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nFastest model: {sorted_models[0]} ({sorted_times[0]:.2f}s)")
print(f"Slowest model: {sorted_models[-1]} ({sorted_times[-1]:.2f}s)")

## 7. Performance vs Training Time Trade-off

In [ ]:
# Scatter plot: F1-Score vs Training Time
fig, ax = plt.subplots(figsize=(10, 8))

f1_scores = [results[m]['f1_score'] for m in models]
train_times = [results[m]['training_time'] for m in models]

colors = plt.cm.Set1(np.linspace(0, 1, len(models)))

for i, model in enumerate(models):
    ax.scatter(train_times[i], f1_scores[i], s=200, c=[colors[i]], 
              label=model, edgecolors='black', linewidths=2)
    ax.annotate(model, (train_times[i], f1_scores[i]), 
               textcoords="offset points", xytext=(10, 5), fontsize=10)

ax.set_xlabel('Training Time (seconds)', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('Performance vs Training Time Trade-off', fontsize=14, fontweight='bold')

# Add quadrant lines
ax.axhline(y=np.mean(f1_scores), color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=np.mean(train_times), color='gray', linestyle='--', alpha=0.5)

# Add quadrant labels
ax.text(0.02, 0.98, 'Fast & Good', transform=ax.transAxes, fontsize=10,
       verticalalignment='top', fontweight='bold', color='green')
ax.text(0.98, 0.98, 'Slow & Good', transform=ax.transAxes, fontsize=10,
       verticalalignment='top', horizontalalignment='right', fontweight='bold', color='orange')
ax.text(0.02, 0.02, 'Fast & Poor', transform=ax.transAxes, fontsize=10,
       verticalalignment='bottom', fontweight='bold', color='red')

ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/performance_vs_time.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Radar Chart: Multi-Metric Comparison

In [ ]:
# Radar chart for comprehensive comparison
from math import pi

# Prepare data
categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
N = len(categories)

# Create angles for radar chart
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the loop

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

colors = plt.cm.Set1(np.linspace(0, 1, len(models)))

for i, model in enumerate(models):
    values = [results[model]['accuracy'], results[model]['precision'],
              results[model]['recall'], results[model]['f1_score']]
    values += values[:1]  # Complete the loop
    
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Multi-Metric Comparison (Radar Chart)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('../results/plots/radar_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Model Ranking

In [ ]:
# Create comprehensive ranking
ranking_df = pd.DataFrame({
    'Model': models,
    'Accuracy': [results[m]['accuracy'] for m in models],
    'Precision': [results[m]['precision'] for m in models],
    'Recall': [results[m]['recall'] for m in models],
    'F1-Score': [results[m]['f1_score'] for m in models],
    'Training Time': [results[m]['training_time'] for m in models],
})

# Add ranks for each metric
ranking_df['F1 Rank'] = ranking_df['F1-Score'].rank(ascending=False).astype(int)
ranking_df['Speed Rank'] = ranking_df['Training Time'].rank(ascending=True).astype(int)

# Calculate overall score (weighted average of ranks)
ranking_df['Overall Rank'] = (ranking_df['F1 Rank'] * 0.7 + ranking_df['Speed Rank'] * 0.3)
ranking_df = ranking_df.sort_values('Overall Rank')

print("\n" + "=" * 80)
print("MODEL RANKING (Weighted: 70% F1-Score, 30% Speed)")
print("=" * 80)
print(ranking_df[['Model', 'F1-Score', 'Training Time', 'F1 Rank', 'Speed Rank', 'Overall Rank']].to_string(index=False))
print("=" * 80)

## 10. Key Insights and Recommendations

In [ ]:
# Find best and worst performers
best_f1_model = max(results.items(), key=lambda x: x[1]['f1_score'])
best_accuracy_model = max(results.items(), key=lambda x: x[1]['accuracy'])
fastest_model = min(results.items(), key=lambda x: x[1]['training_time'])
best_precision_model = max(results.items(), key=lambda x: x[1]['precision'])
best_recall_model = max(results.items(), key=lambda x: x[1]['recall'])

print("\n" + "=" * 70)
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("=" * 70)

print("\n1. BEST PERFORMERS:")
print(f"   - Best F1-Score: {best_f1_model[0]} ({best_f1_model[1]['f1_score']:.4f})")
print(f"   - Best Accuracy: {best_accuracy_model[0]} ({best_accuracy_model[1]['accuracy']:.4f})")
print(f"   - Best Precision: {best_precision_model[0]} ({best_precision_model[1]['precision']:.4f})")
print(f"   - Best Recall: {best_recall_model[0]} ({best_recall_model[1]['recall']:.4f})")
print(f"   - Fastest Training: {fastest_model[0]} ({fastest_model[1]['training_time']:.2f}s)")

print("\n2. MODEL-SPECIFIC INSIGHTS:")
print("   - Logistic Regression: Good baseline, fast and interpretable")
print("   - Naive Bayes: Fastest to train, works well with text data")
print("   - Random Forest: Good performance, provides feature importance")
print("   - SVM: Strong performance on text classification")
print("   - LSTM: Captures sequential patterns, requires more training time")

print("\n3. RECOMMENDATIONS:")
print(f"   - For Production: {best_f1_model[0]} (best F1-Score)")
print(f"   - For Quick Prototyping: {fastest_model[0]} (fastest)")
print("   - For High Precision (avoid false positives): Use precision-optimized model")
print("   - For High Recall (catch all spam): Use recall-optimized model")

print("\n4. TRADE-OFFS:")
print("   - Traditional ML models are faster but may miss complex patterns")
print("   - Deep learning (LSTM) captures context but requires more resources")
print("   - Ensemble methods (Random Forest) balance performance and robustness")

print("=" * 70)

## 11. Summary Table

In [ ]:
# Create final summary
summary = f"""
{'='*70}
FINAL SUMMARY: SMS SPAM DETECTION CASE STUDY
{'='*70}

DATASET:
- SMS Spam Collection Dataset
- ~5,500 messages (87% ham, 13% spam)

MODELS IMPLEMENTED:
1. Logistic Regression - Linear baseline
2. Naive Bayes - Probabilistic classifier
3. Random Forest - Ensemble method
4. SVM - Margin-based classifier
5. LSTM - Deep learning approach

EVALUATION METRICS:
- Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix

KEY FINDINGS:
- Best Overall Model (F1): {best_f1_model[0]} ({best_f1_model[1]['f1_score']:.4f})
- Fastest Model: {fastest_model[0]} ({fastest_model[1]['training_time']:.2f}s)
- All models achieved >90% accuracy on this dataset

CONCLUSION:
For SMS spam detection, {best_f1_model[0]} provides the best balance
between precision and recall, making it the recommended choice for
production deployment.

{'='*70}
"""

print(summary)

# Save summary to file
with open('../results/metrics/summary.txt', 'w') as f:
    f.write(summary)

print("Summary saved to '../results/metrics/summary.txt'")

---

## Conclusion

This comparative analysis demonstrates the effectiveness of various machine learning approaches for SMS spam detection:

1. **All models achieved high accuracy** (>90%), indicating that spam detection is a well-suited task for ML

2. **Traditional ML models** (Logistic Regression, Naive Bayes, SVM) performed competitively with much faster training times

3. **The LSTM model** captured sequential patterns but required significantly more training time

4. **For practical applications**, the choice of model depends on:
   - Required accuracy level
   - Available computational resources
   - Real-time vs batch processing needs
   - Importance of interpretability

---

**Project completed successfully!**

See `REPORT.md` for the complete case study documentation.